# Idempotency keys: stop a retry from booking the same scan twice

**Scenario:** a hospital's triage assistant books urgent imaging slots. A radiologist finds one
stroke patient in two CT slots: the agent asked once, and the runtime booked twice.

The fix works like a numbered ticket at a deli counter: the same ticket never gets a second
sandwich.

### What you will learn

- Explain how a correct retry can still repeat an action.
- Send an idempotency key with each request, so the backend books once per key.
- Keep the keys in a durable ledger, so a restart cannot repeat a booking.

## How your code reports a tool result to the model

Your code reports a tool's result by appending one message to the conversation.

| Field | Value | Why |
|---|---|---|
| `role` | `"tool"` | On OpenAI shaped APIs. Anthropic puts the result in a `user` message instead |
| `tool_call_id` | the `id` from the request | How the model matches result to request |
| `content` | a string | Not an object. Serialise it yourself |

The model sees only that string, never your side effects.

### Step 1: Book the scan and retry when the network fails

![Book the scan and retry when the network fails](images/idempotency-step-1.svg)

Each piece is reasonable on its own, which makes this bug easy to ship.

## What a duplicate booking costs the hospital

Each duplicate booking costs the hospital three things at once.

```
harm = duplicate bookings x (wasted slot + patient recalled + radiologist hour)
```

## A lost reply turns one booking into two

The booking backend below times out once, the way a flaky network does.

In [1]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("01-stateful-agent-runtime/03-capstone-actions-that-survive")

BOOKED = []          # stands in for the scheduling system
FAIL_ONCE = {"left": 1}


def book_scan(patient_id, modality, urgency):
    """Book a slot. Fails the first time, the way a flaky network does."""
    if FAIL_ONCE["left"] > 0:
        FAIL_ONCE["left"] -= 1
        raise TimeoutError("scheduler did not respond")
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    return {"slot": f"SLOT-{len(BOOKED):03d}"}

Next, the model asks for one urgent CT scan.

In [2]:
BOOK_TOOL = {"type": "function", "function": {
    "name": "book_scan",
    "description": "Book an imaging slot for a patient.",
    "parameters": {"type": "object", "properties": {
        "patient_id": {"type": "string"},
        "modality": {"type": "string", "enum": ["ct", "mri", "xray"]},
        "urgency": {"type": "string", "enum": ["routine", "urgent"]}},
        "required": ["patient_id", "modality", "urgency"],
        "additionalProperties": False}}}

reply = client.chat.completions.create(
    model=model_for("default"), max_tokens=300, tools=[BOOK_TOOL],
    messages=[{"role": "system", "content": "You triage radiology referrals."},
              {"role": "user", "content": "Patient P-4471, suspected stroke, needs a CT now."}])

call = reply.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)
print(f"model asked for: {call.function.name}({args})")
print(f"call id        : {call.id}")

model asked for: book_scan({'urgency': 'urgent', 'modality': 'ct', 'patient_id': 'P-4471'})
call id        : tool_book_scan_o3Qm7OjLgHlyekxWfSsw


Most runtimes ship a retry loop like this, which remembers no earlier attempt.

In [3]:
def run_with_retry(name, args, attempts=3):
    """Retry a flaky tool. Nothing here remembers a previous attempt."""
    for attempt in range(attempts):
        try:
            return book_scan(**args)
        except TimeoutError:
            print(f"  attempt {attempt + 1} timed out, retrying")
    raise RuntimeError("gave up")


result = run_with_retry(call.function.name, args)
print(f"\nbooked: {result}")
print(f"rows in the scheduler: {len(BOOKED)}")

  attempt 1 timed out, retrying

booked: {'slot': 'SLOT-001'}
rows in the scheduler: 1


One row, because that timeout came before the write. Next, the write lands and the reply is lost.

In [4]:
BOOKED.clear()


def book_scan_lossy(patient_id, modality, urgency, drop_reply=True):
    """Writes, then loses the response. The caller cannot tell."""
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    if drop_reply and len(BOOKED) == 1:
        raise TimeoutError("scheduler did not respond")
    return {"slot": f"SLOT-{len(BOOKED):03d}"}


for attempt in range(2):
    try:
        book_scan_lossy(**args)
        break
    except TimeoutError:
        print(f"  attempt {attempt + 1} timed out, retrying")

print(f"\nrows in the scheduler: {len(BOOKED)}")
assert len(BOOKED) == 1, f"patient booked {len(BOOKED)} times for one request"

  attempt 1 timed out, retrying

rows in the scheduler: 2


AssertionError: patient booked 2 times for one request

### Step 2: The retry books the same patient twice

![The retry books the same patient twice](images/idempotency-step-2.svg)

The row is written, the reply is lost, and the retry writes it again.

## Why nothing stopped the second booking

A timeout cannot say whether the write landed, and nothing recorded the first attempt, so the retry
looked new.

### Step 3: The request already carried a stable id

![The request already carried a stable id](images/idempotency-step-3.svg)

The `tool_call_id` names one action and stays the same on every retry.

## Check an idempotency key where the write happens

The key must travel with the request, because a ledger written after the call never sees a lost
reply. The scheduler checks it, which makes the booking **idempotent**: safe to run twice, because
the second run changes nothing.

### Step 4: The scheduler checks the key before booking

![The scheduler checks the key before booking](images/idempotency-step-4.svg)

A retry with the same key gets the first result back, not a second row.

In [5]:
SCHEDULER_KEYS = {}      # lives with the scheduler, not with the agent


def book_scan_idempotent(patient_id, modality, urgency, idempotency_key,
                         drop_reply=False):
    """Book at most once per key. The dedupe happens before the write."""
    if idempotency_key in SCHEDULER_KEYS:
        return SCHEDULER_KEYS[idempotency_key]
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    outcome = {"slot": f"SLOT-{len(BOOKED):03d}"}
    SCHEDULER_KEYS[idempotency_key] = outcome
    if drop_reply:
        raise TimeoutError("scheduler did not respond")
    return outcome

Every attempt now sends the same key over the same broken network.

In [6]:
BOOKED.clear()
SCHEDULER_KEYS.clear()

for attempt in range(2):
    try:
        # Reply is dropped on the first attempt, after the row is written.
        outcome = book_scan_idempotent(**args, idempotency_key=call.id,
                                       drop_reply=(attempt == 0))
        print(f"  attempt {attempt + 1} returned {outcome}")
        break
    except TimeoutError:
        print(f"  attempt {attempt + 1} timed out, retrying with the same key")

print(f"\nrows in the scheduler: {len(BOOKED)}")
print(f"before the fix: 2 rows for one request")
print(f"after the fix : {len(BOOKED)} row for one request")

  attempt 1 timed out, retrying with the same key
  attempt 2 returned {'slot': 'SLOT-001'}

rows in the scheduler: 1
before the fix: 2 rows for one request
after the fix : 1 row for one request


Hashing the arguments with the call id keeps the key stable across retries.

In [7]:
import hashlib


def idempotency_key(call_id, args):
    """Stable for one intended action, and unchanged across retries."""
    payload = json.dumps(args, sort_keys=True)
    return f"{call_id}:{hashlib.sha256(payload.encode()).hexdigest()[:12]}"

This result message is all the model ever sees of the booking.

In [8]:
def tool_result_message(call_id, outcome):
    """What the model gets back. The id is how it matches this to its request."""
    return {"role": "tool",
            "tool_call_id": call_id,
            "content": json.dumps(outcome)}


key = idempotency_key(call.id, args)
message = tool_result_message(call.id, SCHEDULER_KEYS[call.id])
print(f"key    : {key}")
print(f"message: {message}")

key    : tool_book_scan_o3Qm7OjLgHlyekxWfSsw:9a506228a02c
message: {'role': 'tool', 'tool_call_id': 'tool_book_scan_o3Qm7OjLgHlyekxWfSsw', 'content': '{"slot": "SLOT-001"}'}


### Step 5: Store the keys in a durable ledger

![Store the keys in a durable ledger](images/idempotency-step-5.svg)

A key held only in memory is lost on restart, so the ledger goes in a file.

In [9]:
import pathlib

STATE_FILE = pathlib.Path("runtime-state.json")


def load_ledger():
    """The ledger a restart can still read."""
    if STATE_FILE.is_file():
        return json.loads(STATE_FILE.read_text())
    return {}

The durable version books once per key, across restarts and retries.

In [10]:
def book_durable(patient_id, modality, urgency, key):
    """Book at most once per key, across restarts as well as retries."""
    ledger = load_ledger()
    if key in ledger:
        return ledger[key], "replayed"
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    ledger[key] = {"slot": f"SLOT-{len(BOOKED):03d}"}
    STATE_FILE.write_text(json.dumps(ledger, indent=2))
    return ledger[key], "executed"

The second call below runs with nothing in memory, exactly like a restart.

In [11]:
BOOKED.clear()
STATE_FILE.unlink(missing_ok=True)

first, how_first = book_durable(**args, key=key)
second, how_second = book_durable(**args, key=key)      # after a restart

print(f"first call : {first} ({how_first})")
print(f"after restart: {second} ({how_second})")
print(f"rows in the scheduler: {len(BOOKED)}")

first call : {'slot': 'SLOT-001'} (executed)
after restart: {'slot': 'SLOT-001'} (replayed)
rows in the scheduler: 1


The cells above printed every row.

| What happened | Rows in the scheduler |
|---|---|
| A lost reply, then a retry with no key | 2 |
| A lost reply, then a retry with the key | 1 |
| A booking, then a restart, with the durable ledger | 1 |

## A test that fails if one action books twice

One test covers retries and restarts, because the ledger cannot tell them apart.

### Step 6: Test five attempts against one booking key

![Test five attempts against one booking key](images/idempotency-step-6.svg)

A second booking of the same patient fails the build.

In [12]:
def test_one_action_happens_once():
    BOOKED.clear()
    STATE_FILE.unlink(missing_ok=True)
    for _ in range(5):                       # retries and restarts alike
        book_durable("P-1", "ct", "urgent", key="same-key")
    assert len(BOOKED) == 1, f"booked {len(BOOKED)} times for one key"


test_one_action_happens_once()
STATE_FILE.unlink(missing_ok=True)
print("gate holds: five attempts across restarts, one booking")

gate holds: five attempts across restarts, one booking


### Enterprise exploration

- A file works for one process, so where does the ledger live across four replicas?
- How long do keys live, and what happens to a retry after its key expires?
- What is the compliance exposure of a duplicate patient record found late?

### Key terms and traps

- **Idempotency key**: an id sent with a request, so the backend acts on it at most once.
- **Durable ledger**: the key store, which must outlive retries and restarts.
- **Trap**: a timeout means the reply was lost, not that the write failed.